In [29]:
import json
import re
import tempfile

from PIL import ImageDraw
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
from dotenv import load_dotenv
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [34]:
from dotenv import load_dotenv
def authenticate():
    '''
    Authenticate
    Authenticates your credentials and creates a client.
    '''
    load_dotenv()
    VISION_KEY = "0iBXNB7UR8uDpjmzScO2TI5W0prTHAH86ISmGYXsA37UXGHmm7yhJQQJ99BCACi5YpzXJ3w3AAAFACOGopTW"
    VISION_ENDPOINT = "https://rares.cognitiveservices.azure.com/"
    subscription_key ="VISION_KEY"
    endpoint ="VISION_ENDPOINT"
    credentials = CognitiveServicesCredentials(subscription_key)
    computervision_client = ComputerVisionClient(endpoint, credentials)
    '''
    END - Authenticate
    '''
    return computervision_client

In [36]:
computervision_client = authenticate()

In [50]:
def get_image_ocr_result(image_path: str, language: str = None):
    with open(image_path, "rb") as img:
        read_response = computervision_client.read_in_stream(
            image=img,
            mode="Printed",
            raw=True,
            language=language
        )
    if 'Operation-Location' not in read_response.headers:
        raise KeyError("The response does not contain 'Operation-Location' in headers.")

    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)
    return read_result

In [51]:
def get_text_of_file(image_path, language=None):
    read_result = get_image_ocr_result(image_path, language)
    # Print the detected text, line by line
    result = ""
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                result += line.text
                result += " "
    return result

In [54]:
get_text_of_file('test2.jpeg')

KeyError: 'Endpoint'